In [4]:
import pandas as pd
import numpy as np

# Load integrated dataset
df = pd.read_csv("data/processed/cleaned_data.csv")

df.head()


,Material ID,Packaging Type,Material Type,Suitable Product Categories,Recommended Packaging Use Cases,Supplier Region,Recyclability (%),Recyclability Category,Recycled Content (%),Reusability (%),...,CO2 Emission per kg (estimated),Waste Reduction Impact (%),Sustainability Target Progress (%),Load Handling Score,Moisture Resistance Score,Thermal Resistance Score,Cost per Unit (USD),Annual Usage (units),Total Material Weight (tons),Supplier Sustainability Compliance (%)
0,MAT_0001,Cardboard Boxes,Cardboard,"E-commerce, Food & Beverage, Consumer Goods, A...",Last-mile delivery and primary e-commerce pack...,EMEA,98.0,High,79.0,49.0,...,0.54,61.0,88.0,6.0,5.0,4.0,2.24,101260.0,790.0,85.0
1,MAT_0002,Protective Fillers (Paper/Biodegradable),Paper/Bio-Based,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,APAC,100.0,High,93.0,31.0,...,0.32,68.0,98.0,3.0,3.0,3.0,1.82,73961.0,545.0,88.0
2,MAT_0003,Steel Racks & Containers,Steel,"Heavy Industrial Components, High-Security Goods","Secure, high-load international shipping and l...",AMERICAS,87.0,High,78.0,100.0,...,3.25,85.0,79.0,8.0,9.0,9.0,25.00,14649.0,4979.0,88.0
3,MAT_0004,Protective Fillers (Paper/Biodegradable),Paper/Bio-Based,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,ROW,100.0,High,89.0,31.0,...,0.31,69.0,87.0,1.0,1.0,4.0,2.43,75036.0,656.0,81.0
4,MAT_0005,Protective Fillers (Paper/Biodegradable),Paper/Bio-Based,"Fragile Items, Cosmetics, Pharmaceuticals, Int...",Void-fill and cushioning for fragile products,LATAM,100.0,High,92.0,28.0,...,0.31,61.0,89.0,3.0,2.0,3.0,1.69,69209.0,549.0,81.0


In [5]:
# Missing values
df.isna().sum().sort_values(ascending=False)

Material ID                               0
Carbon Footprint (kg CO2/unit)            0
Total Material Weight (tons)              0
Annual Usage (units)                      0
Cost per Unit (USD)                       0
Thermal Resistance Score                  0
Moisture Resistance Score                 0
Load Handling Score                       0
Sustainability Target Progress (%)        0
Waste Reduction Impact (%)                0
CO2 Emission per kg (estimated)           0
End-of-Life Disposal (%)                  0
Packaging Type                            0
Biodegradation Time (days)                0
Reusability (%)                           0
Recycled Content (%)                      0
Recyclability Category                    0
Recyclability (%)                         0
Supplier Region                           0
Recommended Packaging Use Cases           0
Suitable Product Categories               0
Material Type                             0
Supplier Sustainability Complian

In [6]:
df["recommended_material"] = df["Material Type"]

In [7]:
df = df.drop(columns=["Material Type"])

In [ ]:
df = df.drop(columns=["Material ID"])

In [10]:
from sklearn.preprocessing import MinMaxScaler

score_cols = [
    "Carbon Footprint (kg CO2/unit)",
    "CO2 Emission per kg (estimated)",
    "Biodegradation Time (days)",
    "Recyclability (%)"
]

scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[score_cols])

df["sustainability_score"] = (
    (1 - scaled[:, 0]) * 0.35 +
    (1 - scaled[:, 1]) * 0.25 +
    (1 - scaled[:, 2]) * 0.20 +
    scaled[:, 3] * 0.20
) * 100


In [11]:
features_list = [

    # Material Performance
    "Carbon Footprint (kg CO2/unit)",
    "CO2 Emission per kg (estimated)",
    "Biodegradation Time (days)",
    "Recyclability (%)",
    "Recycled Content (%)",
    "Reusability (%)",
    "End-of-Life Disposal (%)",
    "Waste Reduction Impact (%)",
    "Sustainability Target Progress (%)",
    "Load Handling Score",
    "Moisture Resistance Score",
    "Thermal Resistance Score",

    # Cost & Usage
    "Cost per Unit (USD)",
    "Annual Usage (units)",
    "Total Material Weight (tons)",

    # Supply Chain
    "Supplier Region",
    "Supplier Sustainability Compliance (%)",

    # Packaging Context
    "Packaging Type",
    "Suitable Product Categories",
    "Recommended Packaging Use Cases",
    "Recyclability Category"
]


In [12]:
X = df[features_list]
y = df["recommended_material"]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (404, 21)
y shape: (404,)


In [14]:
X.to_csv("data/processed/X_raw.csv", index=False)
y.to_csv("data/processed/y_raw.csv", index=False)

##  Preprocessing Pipeline

In [ ]:
products = pd.read_csv("data/raw/EcoPackAI_dataset.csv")
materials = pd.read_csv("data/raw/product_dataset.csv")

In [4]:
products['key'] = 1
materials['key'] = 1

df = products.merge(materials, on='key').drop('key', axis=1)


In [5]:
df.to_csv("data/processed/integrated_dataset.csv", index=False)

In [6]:
excluded_columns = [
    "Material ID",
    "product_id",
    "product_name",
    "Suitable Product Categories",
    "Recommended Packaging Use Cases"
]

In [7]:
numeric_features = [
    # Product
    "product_weight_kg",
    "fragility_index",

    # Material sustainability
    "Recyclability (%)",
    "Recycled Content (%)",
    "Reusability (%)",
    "Biodegradation Time (days)",
    "End-of-Life Disposal (%)",
    "Carbon Footprint (kg CO2/unit)",
    "CO2 Emission per kg (estimated)",
    "Waste Reduction Impact (%)",
    "Sustainability Target Progress (%)",

    # Material performance
    "Load Handling Score",
    "Moisture Resistance Score",
    "Thermal Resistance Score",

    # Cost & usage
    "Cost per Unit (USD)",
    "Annual Usage (units)",
    "Total Material Weight (tons)",
    "Supplier Sustainability Compliance (%)"
]


In [8]:
categorical_features = [
    # Product
    "category",
    "shipping_type",

    # Material
    "Packaging Type",
    "Material Type",
    "Recyclability Category",
    "Supplier Region"
]


In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=excluded_columns)

X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [11]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


In [12]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)


In [13]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [ ]:
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [16]:
import joblib

joblib.dump(
    preprocessor,
    "models/preprocessing/preprocessing_pipeline.pkl"
)


['models/preprocessing/preprocessing_pipeline.pkl']

In [17]:
sample_df = pd.DataFrame(X_train_transformed[:10])

sample_df.to_csv(
    "data/model_ready/sample_transformed.csv",
    index=False
)


## Train/Test Split & Cross-Validation

In [18]:
df["stratify_key"] = (
    df["category"].astype(str) + "_" +
    df["Material Type"].astype(str) + "_" +
    df["shipping_type"].astype(str)
)

In [19]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["stratify_key"])
y = df[["Cost per Unit (USD)", "Carbon Footprint (kg CO2/unit)"]]  # evaluation targets

X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42,
    stratify=df["stratify_key"]
)


In [20]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [24]:
X_train["category"].value_counts(normalize=True)
X_train["Material Type"].value_counts(normalize=True)
X_train["shipping_type"].value_counts(normalize=True)


shipping_type
Road    0.618665
Air     0.310002
Sea     0.071333
Name: proportion, dtype: float64

## preprocessing product dataset

In [2]:
import pandas as pd
product_df = pd.read_csv("data/raw/product_dataset.csv")

In [3]:
# Numeric columns
numeric_cols = ["product_weight_kg", "fragility_index"]

# Fill numeric missing values with median
product_df[numeric_cols] = product_df[numeric_cols].fillna(product_df[numeric_cols].median())

# Categorical columns
categorical_cols = ["category", "shipping_type"]

# Fill categorical missing values with "Unknown"
product_df[categorical_cols] = product_df[categorical_cols].fillna("Unknown")


In [4]:
# Convert numeric columns to float
product_df[numeric_cols] = product_df[numeric_cols].astype(float)

# Convert categorical columns to string
product_df[categorical_cols] = product_df[categorical_cols].astype(str)


In [5]:
# One-Hot Encode category and shipping_type
product_df = pd.get_dummies(product_df, columns=categorical_cols, drop_first=True)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
product_df[numeric_cols] = scaler.fit_transform(product_df[numeric_cols])


In [7]:
product_df.to_csv("data/model_ready/product_cleaned.csv", index=False)

## Combine (Integrate) datasets


In [3]:
import pandas as pd

product_df = pd.read_csv("data/model_ready/product_cleaned_encoded.csv")
material_df = pd.read_csv("data/model_ready/materials_cleaned_encoded.csv")


In [6]:
import pandas as pd

# Define compatibility function
def is_compatible(product, material):
    # Example logic: product weight <= material load handling
    # and fragility <= material moisture resistance
    return (product['product_weight_kg'] <= material['Load Handling Score']) and \
           (product['fragility_index'] <= material['Moisture Resistance Score'])

# Generate integrated dataset
integrated_rows = []

for _, p in product_df.iterrows():
    for _, m in material_df.iterrows():
        if is_compatible(p, m):
            combined = pd.concat([p, m])
            integrated_rows.append(combined)

integrated_dataset = pd.DataFrame(integrated_rows)

target_cols = [
    'Cost per Unit (USD)',
    'Carbon Footprint (kg CO2/unit)'
]

# Safety check
missing = [c for c in target_cols if c not in integrated_dataset.columns]
if missing:
    raise ValueError(f"Missing target columns: {missing}")

feature_cols = [c for c in integrated_dataset.columns if c not in target_cols]

X_raw = integrated_dataset[feature_cols]
Y_raw = integrated_dataset[target_cols]

print("X shape:", X_raw.shape)
print("Y shape:", Y_raw.shape)


# Save outputs
integrated_dataset.to_csv("data/processed/integrated_dataset.csv", index=False)
X_raw.to_csv("data/processed/X_raw.csv", index=False)
Y_raw.to_csv("data/processed/Y_raw.csv", index=False)

print("✅ Integration complete. Files saved:")
print(" - data/processed/integrated_dataset.csv")
print(" - data/processed/X_raw.csv")
print(" - data/processed/Y_raw.csv")


X shape: (378903, 57)
Y shape: (378903, 2)
✅ Integration complete. Files saved:
 - data/processed/integrated_dataset.csv
 - data/processed/X_raw.csv
 - data/processed/Y_raw.csv


## Train Baseline Models

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [8]:
X = pd.read_csv("data/processed/X_raw.csv")
Y = pd.read_csv("data/processed/Y_raw.csv")

In [ ]:
y_cost = Y['Cost per Unit (USD)']
y_co2  = Y['Carbon Footprint (kg CO2/unit)']

In [14]:
X_clean = X.drop(columns=[
    'product_name',
    'Material ID'
], errors='ignore')


In [15]:
X_train, X_test, y_cost_train, y_cost_test = train_test_split(
    X_clean, y_cost, test_size=0.2, random_state=42
)

_, _, y_co2_train, y_co2_test = train_test_split(
    X_clean, y_co2, test_size=0.2, random_state=42
)

In [11]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
    'r2': 'r2'
}

In [16]:
lr = LinearRegression()

cv_cost = cross_validate(
    lr, X_train, y_cost_train,
    cv=cv, scoring=scoring, return_train_score=False
)

lr.fit(X_train, y_cost_train)
y_cost_pred = lr.predict(X_test)

In [17]:
dt = DecisionTreeRegressor(random_state=42)

cv_co2 = cross_validate(
    dt,
    X_train,
    y_co2_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

dt.fit(X_train, y_co2_train)
y_co2_pred = dt.predict(X_test)


In [20]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def summarize_metrics(model_name, target, cv_results, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    return {
        "model": model_name,
        "target": target,
        "cv_mae": -cv_results['test_mae'].mean(),
        "cv_rmse": np.sqrt(-cv_results['test_rmse'].mean()),
        "test_mae": mean_absolute_error(y_true, y_pred),
        "test_rmse": rmse,
        "test_r2": r2_score(y_true, y_pred)
    }


In [21]:
metrics = []

metrics.append(
    summarize_metrics(
        "Linear Regression",
        "Cost per Unit (USD)",
        cv_cost,
        y_cost_test,
        y_cost_pred
    )
)

metrics.append(
    summarize_metrics(
        "Decision Tree Regressor",
        "Carbon Footprint (kg CO2/unit)",
        cv_co2,
        y_co2_test,
        y_co2_pred
    )
)


In [22]:
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("ml/metrics/baseline_metrics.csv", index=False)

metrics_df


,model,target,cv_mae,cv_rmse,test_mae,test_rmse,test_r2
0,Linear Regression,Cost per Unit (USD),1.545462e-02,1.447587e-01,1.549085e-02,2.101015e-02,0.994271
1,Decision Tree Regressor,Carbon Footprint (kg CO2/unit),2.975021e-15,7.302491e-08,3.963884e-15,7.425122e-15,1.000000


## Train Random Forest Model for Cost Prediction

In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# -----------------------------
# 1️⃣ Load processed data
# -----------------------------
X_raw = pd.read_csv("data/processed/X_raw.csv")
Y_cost = pd.read_csv("data/processed/Y_raw.csv")['Cost per Unit (USD)']  # Ensure correct column

# Drop non-numeric columns to avoid issues
# Drop non-numeric columns
non_numeric_cols = X_raw.select_dtypes(include=['object']).columns.tolist()
print("Dropping non-numeric columns:", non_numeric_cols)
X_processed = X_raw.drop(columns=non_numeric_cols)


# -----------------------------
# 2️⃣ Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, Y_cost, test_size=0.2, random_state=42
)

# -----------------------------
# 3️⃣ Cross-validation setup
# -----------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# 4️⃣ Random Forest Configuration
# -----------------------------
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# -----------------------------
# 5️⃣ Train Model
# -----------------------------
rf_model.fit(X_train, y_train)

# -----------------------------
# 6️⃣ Evaluate Cross-Validation
# -----------------------------
mae_scores = -cross_val_score(
    rf_model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error'
)
rmse_scores = np.sqrt(-cross_val_score(
    rf_model, X_train, y_train, cv=cv, scoring='neg_mean_squared_error'
))

print("CV MAE:", mae_scores.mean())
print("CV RMSE:", rmse_scores.mean())

# -----------------------------
# 7️⃣ Evaluate on Test Set
# -----------------------------
y_pred = rf_model.predict(X_test)

test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)

print("\nTest Metrics:")
print(f"MAE: {test_mae:.4f}")
print(f"RMSE: {test_rmse:.4f}")
print(f"R2: {test_r2:.4f}")

# -----------------------------
# 8️⃣ Save Model
# -----------------------------
os.makedirs("ml/models", exist_ok=True)
joblib.dump(rf_model, "ml/models/rf_cost.joblib")
print("\n✅ Model saved: ml/models/rf_cost.joblib")

# -----------------------------
# 9️⃣ Save Evaluation Metrics
# -----------------------------
metrics = pd.DataFrame([{
    "model": "Random Forest Regressor",
    "target": "Cost per Unit (USD)",
    "cv_mae": mae_scores.mean(),
    "cv_rmse": rmse_scores.mean(),
    "test_mae": test_mae,
    "test_rmse": test_rmse,
    "test_r2": test_r2
}])

os.makedirs("ml/metrics", exist_ok=True)
metrics.to_csv("ml/metrics/rf_cost_metrics.csv", index=False)
print("✅ Metrics saved: ml/metrics/rf_cost_metrics.csv")


Dropping non-numeric columns: ['product_name', 'Material ID']
CV MAE: 5.649954337918133e-15
CV RMSE: 9.952631572165962e-15

Test Metrics:
MAE: 0.0000
RMSE: 0.0000
R2: 1.0000

✅ Model saved: ml/models/rf_cost.joblib
✅ Metrics saved: ml/metrics/rf_cost_metrics.csv


## Train XGBoost Model for CO₂ Emission Prediction


In [32]:
import pandas as pd

# Load features
X = pd.read_csv("data/processed/X_raw.csv")

# Load targets (contains cost + CO2)
y_targets = pd.read_csv("data/processed/y_raw.csv")

# Extract ONLY CO2 target
y_co2 = y_targets["Carbon Footprint (kg CO2/unit)"]


In [33]:
# Drop string/object columns
X = X.select_dtypes(include=["number"])


In [34]:
print(X.shape, y_co2.shape)


(378903, 47) (378903,)


In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_co2,
    test_size=0.2,
    random_state=42
)

In [36]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [37]:
xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = xgb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.0010273705820197196
RMSE: 0.0014809440749848802
R2: 0.9999598350556335


In [39]:
metrics_df = pd.DataFrame([{
    "model": "XGBoost Regressor",
    "target": "Carbon Footprint (kg CO2/unit)",
    "mae": mae,
    "rmse": rmse,
    "r2": r2
}])

metrics_df.to_csv("ml/metrics/co2_metrics.csv", index=False)


In [40]:
import joblib

joblib.dump(xgb_model, "ml/models/xgb_co2.joblib")

['ml/models/xgb_co2.joblib']

In [41]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance.to_csv(
    "ml/reports/feature_importance.csv",
    index=False
)

## Model Explainability


In [3]:
import shap
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path

# Load data
X = pd.read_csv("data/processed/integrated_dataset.csv")
X_model = X.select_dtypes(include="number")


# Load trained model (example)
import joblib
cost_model = joblib.load("ml/models/rf_cost.joblib")


c:\Users\prath\OneDrive\Attachments\Desktop\Infosys Project\clean_repo\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Columns that must NOT go into SHAP
drop_cols = [
    "product_name",
    "Material ID",
    "Cost per Unit (USD)",          # target
    "Carbon Footprint (kg CO2/unit)"  # other target
]

X = X.drop(columns=drop_cols)


In [5]:
X = X.select_dtypes(include=["int64", "float64", "bool"])

In [6]:
X = X[cost_model.feature_names_in_]

In [ ]:
explainer = shap.TreeExplainer(cost_model)
shap_values = explainer.shap_values(X)

In [7]:
out_dir = Path("outputs/explainability")
out_dir.mkdir(parents=True, exist_ok=True)

shap.summary_plot(shap_values, X, show=False)
plt.savefig(out_dir / "shap_summary.png", bbox_inches="tight")
plt.close()

print("✅ SHAP summary plot saved")


NameError: name 'shap_values' is not defined